In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql import Window

GOLD_SCHEMA = "medallionsales.gold"
SILVER_SCHEMA = "medallionsales.silver"
spark.sql("CREATE SCHEMA IF NOT EXISTS medallionsales.gold")

DataFrame[]

## Read Silver
***Silver contains trusted, cleaned and integrated business entities.***
***Gold transforms them into an analytical model.***

In [0]:
silver_customer = spark.read.table(
    f"{SILVER_SCHEMA}.silver_customer"
)

silver_products = spark.read.table(
    f"{SILVER_SCHEMA}.silver_product"
)

silver_sales = spark.read.table(
    f"{SILVER_SCHEMA}.silver_sales"
)

####dimension customer####

In [0]:
dim_customer = (
    silver_customer
    .select(
        "customer_id",
        "customer_key",
        "first_name",
        "last_name",
        "martial_status",
        "gender",
        "create_date",
        "country"
    )
)

In [0]:
dim_customer.show()

+-----------+------------+----------+---------+--------------+------+-----------+-------------+
|customer_id|customer_key|first_name|last_name|martial_status|gender|create_date|      country|
+-----------+------------+----------+---------+--------------+------+-----------+-------------+
|       NULL|       SF566|      NULL|     NULL|          NULL|  NULL|       NULL|         NULL|
|      11000|  AW00011000|       jon|     yang|             m|     m| 2025-10-06|    australia|
|      11001|  AW00011001|    eugene|    huang|             s|     m| 2025-10-06|    australia|
|      11002|  AW00011002|     ruben|   torres|             m|     m| 2025-10-06|    australia|
|      11003|  AW00011003|   christy|      zhu|             s|     f| 2025-10-06|    australia|
|      11004|  AW00011004| elizabeth|  johnson|             s|     f| 2025-10-06|    australia|
|      11005|  AW00011005|     julio|     ruiz|             s|     m| 2025-10-06|    australia|
|      11006|  AW00011006|     janet|  a

In [0]:
w_customer_sk = Window.orderBy("customer_id")

dim_customer = (
    dim_customer
    .withColumn(
        "customer_sk",
        F.row_number().over(w_customer_sk).cast("long")
    )
    .select(
        "customer_sk",
        "customer_id",
        "customer_key",
        "first_name",
        "last_name",
        "martial_status",
        "gender",
        "create_date",
        "country"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Customer key must be unique
duplicate_customers = (
    dim_customer
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_customers.count() > 0:
    duplicate_customers.show()
    raise ValueError(
        "dim_customer contains duplicate customer_id values."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Surrogate key must be unique
duplicate_customer_sk = (
    dim_customer
    .groupBy("customer_sk")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_customer_sk.count() > 0:
    raise ValueError(
        "dim_customer contains duplicate customer_sk values."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


####dimension product####

In [0]:
dim_product = (
    silver_products
    .select(
        "product_key",
        "product_business_key",
        "product_name",
        "category",
        "sub_category",
        "maintenance",
        "product_cost",
        "product_line",
        "start_date",
        "end_date"
    )
    .withColumn(
        "is_current",
        F.col("end_date").isNull()
    )
)

In [0]:
dim_product.show()

+----------------+--------------------+--------------------+-----------+------------+-----------+------------+------------+----------+--------+----------+
|     product_key|product_business_key|        product_name|   category|sub_category|maintenance|product_cost|product_line|start_date|end_date|is_current|
+----------------+--------------------+--------------------+-----------+------------+-----------+------------+------------+----------+--------+----------+
|CO-RF-FR-R92B-58|          FR-R92B-58|hl road frame - b...| components| road frames|        yes|        NULL|           r|  20030701|    NULL|      true|
|CO-RF-FR-R92R-58|          FR-R92R-58|hl road frame - r...| components| road frames|        yes|        NULL|           r|  20030701|    NULL|      true|
| AC-HE-HL-U509-R|           HL-U509-R|sport-100 helmet-...|accessories|     helmets|        yes|          12|           s|  20071228|20110701|     false|
| AC-HE-HL-U509-R|           HL-U509-R|sport-100 helmet-...|accessorie

In [0]:
w_product_sk = Window.orderBy(
    "product_key",
    "start_date",
)

dim_product = (
    dim_product
    .withColumn(
        "product_sk",
        F.row_number().over(w_product_sk).cast("long")
    )
    .select(
        "product_sk",
        "product_key",
        "product_business_key",
        "product_name",
        "category",
        "sub_category",
        "maintenance",
        "product_cost",
        "product_line",
        "start_date",
        "end_date",
        "is_current"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
multiple_current_products = (
    dim_product
    .groupBy("product_key")
    .agg(
        F.sum(
            F.when(F.col("is_current"), 1).otherwise(0)
        ).alias("current_versions")
    )
    .filter(F.col("current_versions") > 1)
)

if multiple_current_products.count() > 0:
    multiple_current_products.show()
    raise ValueError(
        "A product has more than one current SCD2 version."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


####dimension date####

In [0]:
#create dim date
dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit("2000-01-01")),
                F.to_date(F.lit("2050-12-31")),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day",
        "day_of_week",
        "day_name"
    )
)

####fact sales####

In [0]:
sales_base = (
    silver_sales
    .withColumn(
        "_sales_row_id",
        F.monotonically_increasing_id()
    )
)

In [0]:
sales_customer = (
    sales_base.alias("s")
    .join(
        dim_customer.select(
            "customer_sk",
            "customer_id"
        ).alias("c"),
        F.col("s.customer_id") == F.col("c.customer_id"),
        "left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
unmatched_customers = (
    sales_customer
    .filter(F.col("c.customer_sk").isNull())
)

print(
    "Sales rows with unmatched customer:",
    unmatched_customers.count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Sales rows with unmatched customer: 0


In [0]:
if unmatched_customers.count() > 0:
    unmatched_customers.select(
        "customer_id"
    ).distinct().show(100, False)

    raise ValueError(
        "Sales contains customer IDs that do not exist in dim_customer."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
sales_product = (
    sales_base.alias("s")
    .join(
        dim_product.alias("p"),
        how = "left",
        on = (F.col("s.product_number") == F.col("p.product_business_key"))
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
product_match_count = (
    sales_product
    .groupBy("_sales_row_id")
    .count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
unmatched_products = (
    sales_product
    .filter(F.col("p.product_sk").isNull())
)

print(
    "Sales rows with unmatched product:",
    unmatched_products.count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Sales rows with unmatched product: 0


In [0]:
if unmatched_products.count() > 0:
    unmatched_products.select(
        "product_number",
        "order_date"
    ).distinct().show(100, False)

    raise ValueError(
        "Sales contains product keys with no valid SCD2 product version."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
fact_sales_joined = (
    sales_base.alias("s")
    .join(
        dim_customer.alias("c"),
        F.col("s.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .join(
        dim_product.alias("p"),
        (F.col("s.product_number") == F.col("p.product_business_key")) &
        (F.col("s.order_date") >= F.col("p.start_date")) &
        (F.col("s.order_date") <= F.col("p.end_date")),
        "left"
    )
    .join(
        dim_date.alias("od"),
        F.col("s.order_date") == F.col("od.date_key"),
        "left"
    )
    .join(
        dim_date.alias("sd"),
        F.col("s.ship_date") == F.col("sd.date_key"),
        "left"
    )
    .join(
        dim_date.alias("dd"),
        F.col("s.due_date") == F.col("dd.date_key"),
        "left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
fact_sales_joined.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+--------------+-----------+----------+---------+--------+------------+--------+------+--------------------+--------------------+-------------+-----------+-----------+------------+----------+---------+--------------+------+-----------+--------------+----------+----------------+--------------------+--------------------+--------+--------------+-----------+------------+------------+----------+--------+----------+--------+----------+----+-------+-----+----------+------------+---+-----------+---------+--------+----------+----+-------+-----+----------+------------+---+-----------+---------+--------+----------+----+-------+-----+----------+------------+---+-----------+---------+
|order_number|product_number|customer_id|order_date|ship_date|due_date|sales_amount|quantity| price|   _ingest_timestamp|        _source_file|_sales_row_id|customer_sk|customer_id|customer_key|first_name|last_name|martial_status|gender|create_date|       country|product_sk|     product_key|product_busines

In [0]:
fact_sales = (
    fact_sales_joined
    .select(
        F.col("s._sales_row_id").alias("sales_line_key"),

        F.col("s.order_number"),

        F.col("c.customer_sk"),
        F.col("p.product_sk"),

        F.col("od.date_key").alias("order_date_key"),
        F.col("sd.date_key").alias("ship_date_key"),
        F.col("dd.date_key").alias("due_date_key"),

        F.col("s.sales_amount"),
        F.col("s.quantity"),
        F.col("s.price"),

        F.col("p.product_cost").alias("unit_cost")
    )
    .withColumn(
        "estimated_cost",
        F.col("quantity") * F.col("unit_cost")
    )
    .withColumn(
        "estimated_margin",
        F.col("sales_amount") - F.col("estimated_cost")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
fact_sales.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+------------+-----------+----------+--------------+-------------+------------+------------+--------+------+---------+--------------+----------------+
|sales_line_key|order_number|customer_sk|product_sk|order_date_key|ship_date_key|due_date_key|sales_amount|quantity| price|unit_cost|estimated_cost|estimated_margin|
+--------------+------------+-----------+----------+--------------+-------------+------------+------------+--------+------+---------+--------------+----------------+
|             0|     SO43697|      10770|       138|      20101229|     20110105|    20110110|        3578|       1|3578.0|     2171|          2171|            1407|
|             1|     SO43698|      17391|        72|      20101229|     20110105|    20110110|        3400|       1|3400.0|     1912|          1912|            1488|
|             2|     SO43699|      14865|        72|      20101229|     20110105|    20110110|        3400|       1|3400.0|     1912|          1912|            1488|
|   

In [0]:
dim_product.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-------------+--------------------+--------------------+-----------+-----------------+-----------+------------+------------+----------+--------+----------+
|product_sk|  product_key|product_business_key|        product_name|   category|     sub_category|maintenance|product_cost|product_line|start_date|end_date|is_current|
+----------+-------------+--------------------+--------------------+-----------+-----------------+-----------+------------+------------+----------+--------+----------+
|         1|AC-BC-BC-M005|             BC-M005|mountain bottle cage|accessories|bottles and cages|         no|           4|           m|  20130701|    NULL|      true|
|         2|AC-BC-BC-R205|             BC-R205|    road bottle cage|accessories|bottles and cages|         no|           3|           r|  20130701|    NULL|      true|
|         3|AC-BC-WB-H098|             WB-H098|water bottle - 30...|accessories|bottles and cages|         no|           2|           s|  20130701|    NULL|    

In [0]:
dim_customer.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-----------+------------+----------+---------+--------------+------+-----------+---------+
|customer_sk|customer_id|customer_key|first_name|last_name|martial_status|gender|create_date|  country|
+-----------+-----------+------------+----------+---------+--------------+------+-----------+---------+
|          1|       NULL|       SF566|      NULL|     NULL|          NULL|  NULL|       NULL|     NULL|
|          2|      11000|  AW00011000|       jon|     yang|             m|     m| 2025-10-06|australia|
|          3|      11001|  AW00011001|    eugene|    huang|             s|     m| 2025-10-06|australia|
|          4|      11002|  AW00011002|     ruben|   torres|             m|     m| 2025-10-06|australia|
|          5|      11003|  AW00011003|   christy|      zhu|             s|     f| 2025-10-06|australia|
+-----------+-----------+------------+----------+---------+--------------+------+-----------+---------+
only showing top 5 rows


####save dimensions/fact####

In [0]:
gold_tables = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_date": dim_date,
    "fact_sales": fact_sales
}

for table_name, df in gold_tables.items():
    (
        df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{GOLD_SCHEMA}.{table_name}"
        )
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


####views####

In [0]:

##total sales, orders, lines, quantity and average order value for each order 
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_order_summary AS
SELECT
    order_number,
    customer_sk,
    order_date_key,
    ship_date_key,
    due_date_key,

    SUM(sales_amount) AS order_sales,
    SUM(quantity) AS order_quantity,
    COUNT(*) AS order_lines,

    MIN(order_date_key) AS order_date,
    MIN(ship_date_key) AS ship_date,
    MIN(due_date_key) AS due_date
FROM {GOLD_SCHEMA}.fact_sales
GROUP BY
    order_number,
    customer_sk,
    order_date_key,
    ship_date_key,
    due_date_key
""")

DataFrame[]

In [0]:
###KPIS
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_sales_kpis AS
SELECT
    SUM(order_sales) AS total_sales,
    COUNT(*) AS total_orders,
    SUM(order_quantity) AS total_quantity,
    AVG(order_sales) AS average_order_value,
    AVG(order_lines) AS average_lines_per_order
FROM {GOLD_SCHEMA}.vw_order_summary
""")

DataFrame[]

In [0]:
##total_sales, total_quantity, total_orders, average_line_sales over time (year, month,)
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_monthly_sales AS
SELECT
    d.year,
    d.month,
    d.month_name,

    SUM(f.sales_amount) AS total_sales,
    SUM(f.quantity) AS total_quantity,
    COUNT(DISTINCT f.order_number) AS total_orders,
    AVG(f.sales_amount) AS average_line_sales

FROM {GOLD_SCHEMA}.fact_sales f

JOIN {GOLD_SCHEMA}.dim_date d
    ON f.order_date_key = d.date_key

GROUP BY
    d.year,
    d.month,
    d.month_name
""")

DataFrame[]

In [0]:
##higest product 
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_product_performance AS
SELECT
    p.product_business_key,
    p.product_key,
    p.product_name,
    p.category,
    p.sub_category,
    p.product_line,

    SUM(f.quantity) AS units_sold,
    SUM(f.sales_amount) AS total_sales,
    COUNT(DISTINCT f.order_number) AS total_orders,

    AVG(f.price) AS average_selling_price,

    SUM(f.estimated_cost) AS estimated_cost,
    SUM(f.estimated_margin) AS estimated_margin

FROM {GOLD_SCHEMA}.fact_sales f

JOIN {GOLD_SCHEMA}.dim_product p
    ON f.product_sk = p.product_sk

GROUP BY
    p.product_business_key,
    p.product_key,
    p.product_name,
    p.category,
    p.sub_category,
    p.product_line
""")

DataFrame[]

In [0]:
##higest revenue by category
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_category_performance AS
SELECT
    p.category,
    p.sub_category,
    p.product_line,

    SUM(f.quantity) AS units_sold,
    SUM(f.sales_amount) AS total_sales,
    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.estimated_margin) AS estimated_margin

FROM {GOLD_SCHEMA}.fact_sales f

JOIN {GOLD_SCHEMA}.dim_product p
    ON f.product_sk = p.product_sk

GROUP BY
    p.category,
    p.sub_category,
    p.product_line
""")

DataFrame[]

In [0]:
spark.sql(f"""
          select * from {GOLD_SCHEMA}.vw_category_performance
          """).show()

+--------+--------------+------------+----------+-----------+------------+----------------+
|category|  sub_category|product_line|units_sold|total_sales|total_orders|estimated_margin|
+--------+--------------+------------+----------+-----------+------------+----------------+
|   bikes|    road bikes|           r|      2210|    4808994|        2095|         1865267|
|   bikes|mountain bikes|           m|       571|    1406873|         571|          634143|
+--------+--------------+------------+----------+-----------+------------+----------------+



In [0]:
##top customers
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_customer_performance AS
SELECT
    c.customer_id,
    c.customer_key,
    c.first_name,
    c.last_name,
    c.gender,
    c.martial_status,
    c.country,

    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.quantity) AS total_quantity,
    SUM(f.sales_amount) AS total_sales,
    AVG(f.sales_amount) AS average_line_sales

FROM {GOLD_SCHEMA}.fact_sales f

JOIN {GOLD_SCHEMA}.dim_customer c
    ON f.customer_sk = c.customer_sk

GROUP BY
    c.customer_id,
    c.customer_key,
    c.first_name,
    c.last_name,
    c.gender,
    c.martial_status,
    c.country
""")

DataFrame[]

In [0]:
##top countries
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_country_performance AS
SELECT
    c.country,

    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.quantity) AS total_quantity,
    SUM(f.sales_amount) AS total_sales,
    SUM(f.estimated_margin) AS estimated_margin

FROM {GOLD_SCHEMA}.fact_sales f

JOIN {GOLD_SCHEMA}.dim_customer c
    ON f.customer_sk = c.customer_sk

GROUP BY
    c.country
""")

DataFrame[]